### Tools
Models can request to call tools that perform tasks such as fetching data from a database, searching the web, or running code. Tools are pairings of:
1. A schema, including the name of the tool, a description, and/or argument definitions (often a JSON schema)
2. A function or coroutine to execute.

In [1]:
import os
from langchain.chat_models import init_chat_model

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

model = init_chat_model("groq:qwen/qwen3-32b")
response = model.invoke("Why do parrots talk?")
response

Failed to multipart ingest runs: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


AIMessage(content='<think>\nOkay, so the user is asking why parrots talk. Let me start by recalling what I know about parrots. They\'re known for their ability to mimic human speech, right? But why exactly do they do that? Maybe it\'s related to their natural behavior. I remember that parrots are social animals. In the wild, they might imitate sounds to communicate with each other. So in captivity, when they\'re around humans, they might imitate human speech as a way to socialize.\n\nWait, but is there more to it? Their vocal cords are different from humans. I think they use a structure called the syrinx, which allows them to produce a variety of sounds. So their ability to mimic isn\'t just about copying; it\'s also about their vocal anatomy. Maybe they can learn sounds through repetition and reinforcement. Like, when a parrot hears a word and gets a reward, it\'s more likely to repeat it. But some parrots seem to talk without training. Is that just random imitation, or is there a pur

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [3]:
from langchain.tools import tool

@tool
def get_weather(location:str)->str:
    """Get the weather at a location"""
    return f"It's sunny in {location}"


model_with_tools=model.bind_tools([get_weather])
model_with_tools

RunnableBinding(bound=ChatGroq(output_version=None, client=<groq.resources.chat.completions.Completions object at 0x000001FC3E6F29F0>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x000001FC46D247A0>, model_name='qwen/qwen3-32b', model_kwargs={}, groq_api_key=SecretStr('**********'), groq_api_base=None, groq_proxy=None), kwargs={'tools': [{'type': 'function', 'function': {'name': 'get_weather', 'description': 'Get the weather at a location', 'parameters': {'properties': {'location': {'type': 'string'}}, 'required': ['location'], 'type': 'object'}}}]}, config={}, config_factories=[])

In [5]:
response = model_with_tools.invoke("What's the weather like in Boston?")
print(response)

content='' additional_kwargs={'reasoning_content': "Okay, the user is asking about the weather in Boston. I need to use the get_weather function. Let me check the function parameters. It requires a location, which is Boston here. I'll call the function with location set to Boston. Make sure the JSON is correctly formatted with the name and arguments.\n", 'tool_calls': [{'id': 'cjwp34vbb', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]} response_metadata={'token_usage': {'completion_tokens': 86, 'prompt_tokens': 154, 'total_tokens': 240, 'completion_time': 0.152135122, 'prompt_time': 0.006465442, 'queue_time': 0.164751416, 'total_time': 0.158600564, 'completion_tokens_details': {'reasoning_tokens': 62}}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_2bfcc54d36', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logprobs': None} id='lc_run--019dd557-58fb-78c3-a15c-0b52cc4ccb08-0' tool_calls=[{'name': 'get_weather', '

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [7]:
for tool_call in response.tool_calls:
    # View tool calls made by the model
    print(f"Tool Call: {tool_call}")
    print(f"Tool: {tool_call['name']}")
    print(f"Args: {tool_call['args']}")

Tool Call: {'name': 'get_weather', 'args': {'location': 'Boston'}, 'id': 'cjwp34vbb', 'type': 'tool_call'}
Tool: get_weather
Args: {'location': 'Boston'}


### Tool Execution Loops

In [12]:
# Step 1: Model generates tool calls
messages = [{"role": "user", "content": "What's the weather in Boston?"}]
ai_msg = model_with_tools.invoke(messages)
print(messages)
messages.append(ai_msg)
print(messages)

# Step 2: Execute tools and collect results
for tool_call in ai_msg.tool_calls:
    # Execute the tool with the generated arguments
    tool_result = get_weather.invoke(tool_call)
    print("tool result::", tool_result.content)
    messages.append(tool_result)
    print(messages)


# Step 3: Pass results back to model for final response
final_response = model_with_tools.invoke(messages)
print(final_response)
print(final_response.text)
# "The current weather in Boston is 72°F and sunny."

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


[{'role': 'user', 'content': "What's the weather in Boston?"}]
[{'role': 'user', 'content': "What's the weather in Boston?"}, AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'sxeqx2rsq', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 153, 'total_tokens': 247, 'completion_time': 0.144235326, 'prompt_time': 0.006371898, 'queue_time': 0.171381975, 'total_time': 0.150607224, 'completion_tokens_details': {'reasoning_tokens': 70}}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_99d722e776', 'ser

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


content="The weather in Boston is sunny today! 🌞 Let me know if you'd like to check another location." additional_kwargs={'reasoning_content': 'Okay, the user asked, "What\'s the weather in Boston?" I used the get_weather function to check, and the response was, "It\'s sunny in Boston." Now I need to relay that information back to the user in a friendly and clear way. Let me make sure to mention the location and the weather condition. I should also add a bit more detail to make it sound natural, like maybe the temperature or a suggestion. Wait, the function response only says it\'s sunny. So I shouldn\'t add extra info that\'s not provided. Just confirm the sunny weather in Boston. Maybe say something like, "The weather in Boston is sunny today!" and keep it simple. That\'s accurate and matches the tool\'s response.\n'} response_metadata={'token_usage': {'completion_tokens': 176, 'prompt_tokens': 191, 'total_tokens': 367, 'completion_time': 0.393134142, 'prompt_time': 0.014576834, 'que

Failed to send compressed multipart ingest: langsmith.utils.LangSmithError: Failed to POST https://api.smith.langchain.com/runs/multipart in LangSmith API. HTTPError('403 Client Error: Forbidden for url: https://api.smith.langchain.com/runs/multipart', '{"error":"Forbidden"}\n')


In [13]:
messages

[{'role': 'user', 'content': "What's the weather in Boston?"},
 AIMessage(content='', additional_kwargs={'reasoning_content': 'Okay, the user is asking for the weather in Boston. I need to use the get_weather function. The function requires a location parameter. Boston is the location here. So I should call get_weather with location set to "Boston". Let me make sure there are no typos. Everything looks good. I\'ll format the tool call as specified.\n', 'tool_calls': [{'id': 'sxeqx2rsq', 'function': {'arguments': '{"location":"Boston"}', 'name': 'get_weather'}, 'type': 'function'}]}, response_metadata={'token_usage': {'completion_tokens': 94, 'prompt_tokens': 153, 'total_tokens': 247, 'completion_time': 0.144235326, 'prompt_time': 0.006371898, 'queue_time': 0.171381975, 'total_time': 0.150607224, 'completion_tokens_details': {'reasoning_tokens': 70}}, 'model_name': 'qwen/qwen3-32b', 'system_fingerprint': 'fp_99d722e776', 'service_tier': 'on_demand', 'finish_reason': 'tool_calls', 'logpr